In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error

# 1. Load and Clean Data
df = pd.read_csv('AEP_hourly.csv') # Replace with your specific PJM file if different
df['Datetime'] = pd.to_datetime(df['Datetime'])
df = df.sort_values('Datetime').set_index('Datetime')

# 2. Feature Engineering
def create_features(df):
    df = df.copy()
    df['hour'] = df.index.hour
    df['dayofweek'] = df.index.dayofweek
    df['quarter'] = df.index.quarter
    df['month'] = df.index.month
    df['year'] = df.index.year
    df['dayofyear'] = df.index.dayofyear

    # Lag and Rolling Features
    df['lag_24'] = df.iloc[:, 0].shift(24)
    df['rolling_mean_3'] = df.iloc[:, 0].shift(1).rolling(window=3).mean()
    return df

df_features = create_features(df).dropna()

# 3. Chronological Split (Train < 2017, Test >= 2017)
split_date = '2017-01-01'
train = df_features.loc[df_features.index < split_date]
test = df_features.loc[df_features.index >= split_date]

X_cols = ['hour', 'dayofweek', 'quarter', 'month', 'year', 'dayofyear', 'lag_24', 'rolling_mean_3']
y_col = df.columns[0]

X_train, y_train = train[X_cols], train[y_col]
X_test, y_test = test[X_cols], test[y_col]

# 4. Model Training
model = xgb.XGBRegressor(
    n_estimators=1000,
    early_stopping_rounds=50,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=100
)

# 5. Evaluation
preds = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))
mae = mean_absolute_error(y_test, preds)

print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")

[0]	validation_0-rmse:2479.92892	validation_1-rmse:2419.82916
[100]	validation_0-rmse:380.70445	validation_1-rmse:394.85947
[200]	validation_0-rmse:305.92859	validation_1-rmse:328.20596
[300]	validation_0-rmse:274.93490	validation_1-rmse:302.08290
[400]	validation_0-rmse:257.65685	validation_1-rmse:288.12024
[500]	validation_0-rmse:246.11847	validation_1-rmse:279.70600
[600]	validation_0-rmse:238.18412	validation_1-rmse:274.14941
[700]	validation_0-rmse:230.68693	validation_1-rmse:269.81116
[800]	validation_0-rmse:224.33033	validation_1-rmse:266.08993
[900]	validation_0-rmse:219.01359	validation_1-rmse:262.37514
[999]	validation_0-rmse:214.55775	validation_1-rmse:260.27318
RMSE: 260.27
MAE: 196.62


In [3]:
import pandas as pd
import numpy as np

def predict_user_input(model, df_features, X_cols):
    print("\n--- Enter Feature Details for Prediction ---")
    date_str = input("Enter Date and Time (YYYY-MM-DD HH:MM): ")

    try:
        target_dt = pd.to_datetime(date_str)
    except Exception:
        print("Invalid date format.")
        return

    # Extract base time features
    user_features = {
        'hour': target_dt.hour,
        'dayofweek': target_dt.dayofweek,
        'quarter': target_dt.quarter,
        'month': target_dt.month,
        'year': target_dt.year,
        'dayofyear': target_dt.dayofyear
    }

    # Extract historical dependencies from existing data
    lag_target_time = target_dt - pd.Timedelta(hours=24)

    if lag_target_time in df_features.index:
        user_features['lag_24'] = df_features.loc[lag_target_time, df_features.columns[0]]
    else:
        user_features['lag_24'] = df_features['lag_24'].median()

    # Estimate rolling mean based on closest historical hour match
    historical_hour_match = df_features[df_features.index.hour == target_dt.hour]
    if not historical_hour_match.empty:
        user_features['rolling_mean_3'] = historical_hour_match['rolling_mean_3'].mean()
    else:
        user_features['rolling_mean_3'] = df_features['rolling_mean_3'].median()

    # Convert to DataFrame format required by XGBoost
    input_df = pd.DataFrame([user_features])[X_cols]

    # Predict
    prediction = model.predict(input_df)[0]
    print(f"\nPredicted Energy Consumption: {prediction:.2f} MW")

# Call the function (Run this after training your model)
predict_user_input(model, df_features, X_cols)


--- Enter Feature Details for Prediction ---
Enter Date and Time (YYYY-MM-DD HH:MM): 2026-06-03 23:05

Predicted Energy Consumption: 15399.91 MW
